# Stage 10 — Feedback Generator
Uses **Llama 3 (via Groq)** to produce **student-facing feedback**: plain English explanation of what went wrong, why, and how to fix it.

**Input:** incorrect step, previous step, misconception category, correct continuation  
**Output:** a structured feedback string

Prototype here, then copy `generate_feedback()` into `src/pipeline/feedback.py`.

In [1]:
%pip install groq python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env
client = Groq()

# Get your free API key at: https://console.groq.com
# Add to .env: GROQ_API_KEY=your_key_here

## Build the prompt
Key design goals:
- Encouraging, not harsh
- Explains *why* the step is wrong
- Shows the correct path forward
- Adapts depth to error type (computational = brief, conceptual = more explanation)
- Short enough for a student to actually read

In [3]:
SYSTEM_PROMPT = """You are a supportive high school math tutor giving feedback on algebra work.
Adapt your response based on the error type:
- Computational errors (sign, arithmetic): be brief, just point out the slip
- Procedural errors (factorization, zero product): show the correct procedure step by step
- Conceptual errors: explain the underlying rule before showing the fix
- Radical/discriminant errors: show the correct calculation clearly

Rules:
- Under 80 words
- Encouraging tone
- No LaTeX — use ^ for powers (e.g. x^2)
- Use plain language suitable for a high school student

Respond ONLY with a JSON object — no explanation, no markdown:
{"feedback": "<your feedback text>"}"""


def generate_feedback(
    step_prev: str,
    step_wrong: str,
    step_num: int,
    operation: str,
    misconception_category: str,
    misconception_subcategory: str,
    correct_continuation: list,
    retries: int = 3
) -> str:
    """
    Generate student-facing feedback using Llama 3 via Groq.
    Returns: feedback string
    """
    correct_str = " → ".join(correct_continuation)

    user_msg = f"""The student made an error at Step {step_num}.

Previous step (correct): {step_prev}
Student's step (wrong):  {step_wrong}
Operation attempted:     {operation}
Error type:              {misconception_category} — {misconception_subcategory}
Correct continuation:    {correct_str}

Write feedback for the student."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0.3  # slight variation for natural language
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"): raw = raw[4:]
                raw = raw.strip()
            try:
                return json.loads(raw)["feedback"]
            except (json.JSONDecodeError, KeyError):
                return raw  # fallback: return raw text

        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                return "Sorry, feedback could not be generated."

## Test cases

In [4]:
# Test 1: Wrong-sign factorization
fb = generate_feedback(
    step_prev="x^2 - 5x + 6 = 0",
    step_wrong="(x - 2)(x + 3) = 0",
    step_num=2,
    operation="Factorization",
    misconception_category="Procedural Error",
    misconception_subcategory="Incorrect factorization",
    correct_continuation=["(x - 2)(x - 3) = 0", "x = 2 OR x = 3"]
)
print("Test 1 — Wrong sign factorization:")
print(fb)

Test 1 — Wrong sign factorization:
Let's factor correctly: find two numbers that multiply to 6 and add to -5, which are -2 and -3, so it's (x - 2)(x - 3) = 0


In [5]:
# Test 2: Missing root
fb = generate_feedback(
    step_prev="(x - 2)(x - 3) = 0",
    step_wrong="x = 2",
    step_num=3,
    operation="Apply Zero Product Rule",
    misconception_category="Computational Error",
    misconception_subcategory="Missing root",
    correct_continuation=["x - 2 = 0 OR x - 3 = 0", "x = 2 OR x = 3"]
)
print("Test 2 — Missing root:")
print(fb)

Test 2 — Missing root:
Great start, don't forget the other factor, it's x = 2 or x = 3


In [6]:
# Test 3: Radical simplification error (from your dataset: √12 = 12)
fb = generate_feedback(
    step_prev="x = (-(-5) ± √((-5)^2 - 4(1)(-50)))/(2*1)",
    step_wrong="√12 = 12",
    step_num=3,
    operation="Simplify Radical",
    misconception_category="Radical & Simplification Error",
    misconception_subcategory="Incorrect square root simplification",
    correct_continuation=["√225 = 15", "x = (5 ± 15)/2", "x = 10 OR x = -5"]
)
print("Test 3 — Radical simplification error:")
print(fb)

Test 3 — Radical simplification error:
Remember, √ means finding a number that, when multiplied by itself, gives the original value. So, √225 is actually 15, not 12. Let's redo the steps.


In [7]:
# Test 4: Discriminant arithmetic error (from your dataset: 0 - -144 = 145)
fb = generate_feedback(
    step_prev="x = (-(0) ± √((0)^2 - 4(3)(-12)))/(2*3)",
    step_wrong="0 - -144 = 145",
    step_num=3,
    operation="Calculate Discriminant",
    misconception_category="Radical & Simplification Error",
    misconception_subcategory="Discriminant calculation error",
    correct_continuation=["Discriminant = 144", "x = (0 ± 12)/6", "x = 2 OR x = -2"]
)
print("Test 4 — Discriminant error:")
print(fb)

Test 4 — Discriminant error:
Let's recalculate the discriminant: (0)^2 - 4(3)(-12) = 0 + 144 = 144, not 145. This changes the rest of the problem.


## LLM-as-judge: automated feedback quality scoring

In [9]:
# Option A: run test cells first, then score the last fb
score = score_feedback(fb, "Discriminant calculation error", "Discriminant = 144")

# Option B: score a specific feedback string directly
score = score_feedback(
    feedback_text="Great effort! At Step 3, you wrote 0 - -144 = 145, but subtracting a negative means adding: 0 + 144 = 144. The correct discriminant is 144, giving x = (0 ± 12)/6.",
    error_type="Discriminant calculation error",
    correct_step="Discriminant = 144"
)

print("Feedback score:", score)
print(f"Total: {sum(score.values())}/9")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [10]:
JUDGE_PROMPT = """Rate this student math feedback on 3 criteria. Score each 1-3.

1 = poor, 2 = acceptable, 3 = good

Criteria:
- accuracy: does it correctly identify what went wrong?
- clarity: would a high school student understand it?
- actionability: does it show what to do next?

Respond ONLY with JSON: {"accuracy": <1-3>, "clarity": <1-3>, "actionability": <1-3>}"""


def score_feedback(feedback_text: str, error_type: str, correct_step: str) -> dict:
    prompt = f"""{JUDGE_PROMPT}

Feedback: {feedback_text}
Error type: {error_type}
Correct step: {correct_step}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
        raw = raw.strip()
    try:
        return json.loads(raw)
    except:
        return {"accuracy": 0, "clarity": 0, "actionability": 0}


# Score the test feedbacks above
test_cases = [
    ("Incorrect factorization", "(x - 2)(x - 3) = 0"),
    ("Missing root",            "x = 2 OR x = 3"),
    ("Radical simplification",  "√225 = 15"),
    ("Discriminant error",      "Discriminant = 144")
]

feedbacks = [fb]  # add your generated feedbacks here to score them

# Example: score the last generated feedback
score = score_feedback(fb, "Discriminant calculation error", "Discriminant = 144")
print("Feedback score:", score)
print(f"Total: {sum(score.values())}/9")

Feedback score: {'accuracy': 3, 'clarity': 3, 'actionability': 2}
Total: 8/9


## Compare temperature settings

## ✅ Once prompt is good → copy `generate_feedback()` to `src/pipeline/feedback.py`